In [2]:
import cv2
from ultralytics import YOLO
import numpy as np

# 1. Cargar el modelo YOLOv8L
model = YOLO('yolov8l.pt')

# 2. Clases de interés: vehículos + “cell phone”
vehicle_classes = ['car', 'truck', 'bus', 'motorcycle', 'cell phone']
vehicle_class_ids = [i for i, name in model.names.items() if name in vehicle_classes]

# 3. Definición de las 26 celdas de estacionamiento (x1, y1, x2, y2)
parking_spaces = [
    (40, 290, 76, 358), (82, 288, 116, 357), (120, 287, 154, 358), (160, 289, 195, 358),
    (201, 289, 238, 358), (242, 288, 278, 357), (283, 288, 342, 358), (348, 288, 398, 358),
    (404, 287, 457, 356), (462, 287, 512, 356), (520, 287, 569, 358), (576, 286, 614, 357),
    (618, 286, 640, 359), (489, 104, 530, 178), (446, 105, 484, 178), (406, 107, 441, 179),
    (364, 108, 401, 180), (322, 110, 358, 178), (281, 108, 318, 180), (240, 110, 276, 180),
    (204, 112, 234, 179), (126, 112, 157, 184), (89, 114, 120, 180), (44, 113, 80, 184),
    (4, 114, 38, 184), (1, 1, 38, 89)
]

def intersection_area(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    return max(0, xB - xA) * max(0, yB - yA)

# 4. Abrir el video y configurar la ventana
cap = cv2.VideoCapture('test_videos/4.mp4')
ret, frame = cap.read()
if not ret:
    raise Exception('No se pudo leer el video')
h, w = frame.shape[:2]
cv2.namedWindow("Parking Monitor", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Parking Monitor", w, h)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Estado inicial: todas las celdas libres
    occupied = [False] * len(parking_spaces)

    # Detección y seguimiento
    results = model.track(frame, persist=True)

    if results and results[0].boxes.data is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)

        for (x1, y1, x2, y2), cls in zip(boxes, cls_ids):
            # Solo interesan nuestras clases
            if cls not in vehicle_class_ids:
                continue
            # Filtrar por área mínima para no contar 'ruido'
            area = (x2 - x1) * (y2 - y1)
            if area < 100:
                continue

            # Dibujar la caja (sin texto si es cell phone)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
            if model.names[cls] != 'cell phone':
                cv2.putText(frame, model.names[cls], (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            # Marcar celdas ocupadas
            for i, (px1, py1, px2, py2) in enumerate(parking_spaces):
                if intersection_area((x1, y1, x2, y2), (px1, py1, px2, py2)) > 0:
                    occupied[i] = True

    # 5. Dibujar siempre las 26 celdas: verde si libre, rojo si ocupado
    for i, (px1, py1, px2, py2) in enumerate(parking_spaces):
        color = (0, 0, 255) if occupied[i] else (0, 255, 0)
        cv2.rectangle(frame, (px1, py1), (px2, py2), color, 2)
        cv2.putText(frame, str(i+1), (px1, py1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Mostrar
    cv2.imshow("Parking Monitor", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 14 cell phones, 34.5ms
Speed: 1.1ms preprocess, 34.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 33.5ms
Speed: 1.9ms preprocess, 33.5ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 30.7ms
Speed: 1.3ms preprocess, 30.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 18.3ms
Speed: 1.2ms preprocess, 18.3ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 18.0ms
Speed: 1.0ms preprocess, 18.0ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 18.1ms
Speed: 1.0ms preprocess, 18.1ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 19.2ms
Speed: 1.4ms preprocess, 19.2ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 cell phones, 17.8ms
Speed: 1.0ms preprocess, 17.8ms inference

KeyboardInterrupt: 